# PEPO Visualization and Analysis
This notebook provides analysis and visualization for PEPO (Preference Ensemble Policy Optimization) experiments.

## Setup
Import necessary libraries and the `utils` module.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

#apend cwd/notebooks to sys.path
sys.path.append(os.path.join(os.getcwd(), "notebooks"))

import utils
from IPython.display import display, Latex

MODELS = utils.MODELS

## Data Loading
Fetch runs from WandB or load from local cache.

In [ ]:
# Set force_refresh=True to fetch new runs from WandB
df = utils.get_runs_df(force_refresh=False)
print(f"Loaded {len(df)} runs.")

# Experiment 1: Greedy Sampling
Comparing various algorithms (DPO, SFT+DPO, $\chi^2$PO, PEPO) using token-level greedy sampling.

In [ ]:
# Prepare data for all 4 models
dfs = [utils.get_exp1_data(df, model_idx=i) for i in range(len(MODELS))]

# Plot comparison with error bands
utils.plot_multi_model_comparison(
    dfs,
    exclude_algos=['sftdpo','chi2po'],
    x_col='epoch',
    y_col='winrate_initial',
    se_col='standard_error_initial',
    save_path='figures/win_rate_initial.pdf' 
)

### Best Win Rates Summary Table (LaTeX)

In [ ]:
# Get summary of best win rates
summary_df = utils.get_best_winrates(dfs, aggregate_pepo=True)

# Format as LaTeX table
latex_table = utils.format_winrates_latex(summary_df, pivot=True)
print("LaTeX Output:")
print(latex_table)

display(summary_df.head())

# Experiment 2: Rejection Sampling vs Token-Level
Comparing performance across different sampling strategies (Greedy vs. Rejection Sampling variants).

In [ ]:
# Get data for Experiment 2 (epochs 1-5)
df_exp2 = utils.get_exp2_data(df, model_idx=0, epoch_range=(1, 5)) # epoch 4 not yet finished

print("Algorithms found:", df_exp2['algorithm'].unique())

# Plot individual variants
utils.plot_multi_model_comparison(
    [df_exp2],
    aggregate_best=False,
    se_col='standard_error_initial'
)

# Plot aggregated (Best Rejection vs Best Token-Level)
# epoch
utils.plot_multi_model_comparison(
    [df_exp2],
    aggregate_best=True,
    se_col='standard_error_initial'
)

### Rejection Sampling vs Token-Level Comparison Table

In [ ]:
# Generate comparison table for epochs 1-3
comp_table = utils.format_exp2_comparison_table(
    df_exp2, 
    epochs=[1, 2, 3], 
    aggregate_best=True
)
print(comp_table)